# ZimPrep — Exam Paper Extraction Platform

Run this notebook top-to-bottom in Google Colab. It will:
1. Install dependencies (PaddleOCR, PyMuPDF, Qwen2.5-VL, Gradio, ...)
2. Clone / copy the `paper_pipeline` codebase
3. Launch the Gradio UI with a public `share=True` link employees can use in the browser.

**Recommended runtime:** GPU (T4 is fine for OCR + Qwen2.5-VL-3B; A100 for 7B).

In [ ]:
# 1. Install system + Python dependencies
!apt-get -qq install -y poppler-utils libgl1 > /dev/null
!pip -q install --upgrade pip
!pip -q install pymupdf pdfplumber opencv-python-headless pillow numpy tqdm rapidfuzz pydantic
!pip -q install paddlepaddle-gpu paddleocr
!pip -q install --upgrade transformers accelerate sentencepiece einops qwen-vl-utils
!pip -q install gradio>=4.40

In [ ]:
# 2. Bring the pipeline code into the Colab session.
# Option A: clone from your private repo (recommended for production)
#   !git clone https://github.com/<org>/paper_pipeline.git
#   %cd paper_pipeline
#
# Option B: upload the paper_pipeline/ folder via the Files panel, then:
import os, sys, pathlib
PIPELINE_ROOT = '/content/paper_pipeline'  # adjust if you uploaded elsewhere
assert pathlib.Path(PIPELINE_ROOT).exists(), f'Upload paper_pipeline/ to {PIPELINE_ROOT} first.'
if PIPELINE_ROOT not in sys.path:
    sys.path.insert(0, PIPELINE_ROOT)
os.chdir(PIPELINE_ROOT)

In [ ]:
# 3. Sanity check — verify GPU and library imports
import torch
print('CUDA available :', torch.cuda.is_available())
print('Device          :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
from pipeline import run_pipeline, PipelineOptions  # noqa
from ui.app import build_demo, launch  # noqa
print('Imports OK.')

In [ ]:
# 4. Launch the Gradio UI with a public share link.
# Employees can open the printed `*.gradio.live` URL in any browser.
from ui.app import build_demo
demo = build_demo()
demo.queue(max_size=8).launch(share=True, server_name='0.0.0.0')

## Headless / batch use
If you want to run the pipeline programmatically (no UI), use:

```python
from pipeline import run_pipeline, PipelineOptions, save_export
result = run_pipeline('/content/sample.pdf', PipelineOptions(use_vlm=True, use_gpu=True))
out = save_export(result)
print('JSON saved to', out)
```